In [ ]:
import os
import re
import sys
import traceback


print("=" * 60)
print("NEMOTRON LORA v38 — SYMBOLIC-VERIFIED SFT")
print("No external deps — transformers + peft only")
print("=" * 60)

try:
    import kagglehub
    import pandas as pd
    import torch
    from datasets import Dataset
    from peft import LoraConfig, TaskType, get_peft_model
    from transformers import (
        AutoModelForCausalLM,
        AutoTokenizer,
        DataCollatorForLanguageModeling,
        Trainer,
        TrainingArguments,
    )

    print("[1/6] All dependencies loaded (pre-installed)")

    # ── Symbolic Solver ──
    def parse_examples(prompt):
        pairs = []
        for line in prompt.split("\\n"):
            line = line.strip()
            if " -> " in line and not line.startswith(("Here", "Now")):
                parts = line.split(" -> ")
                if len(parts) == 2:
                    pairs.append((parts[0].strip(), parts[1].strip()))
            elif " becomes " in line and not line.startswith("Now"):
                parts = line.split(" becomes ")
                if len(parts) == 2:
                    pairs.append((parts[0].strip(), parts[1].strip()))
        return pairs

    def classify_problem(prompt):
        p = prompt.lower()
        if "bit manipulation" in p: return "bit_manip"
        elif "gravitational" in p or "falling distance" in p: return "gravity"
        elif "unit conversion" in p: return "unit_conversion"
        elif "equation" in p or "transformation" in p: return "equations"
        elif "numeral" in p or "roman" in p: return "numeral"
        elif "encrypt" in p or "cipher" in p: return "encryption"
        return "unknown"

    def extract_test_input(prompt):
        patterns = [
            r"determine the output for:\\s*([0-9a-zA-Z\\s.]+)",
            r"convert the following measurement:\\s*([0-9.]+\\s*m?)",
            r"write the number\\s+([0-9]+)\\s+in the",
            r"decrypt the following text:\\s*([^\\n]+)",
            r"determine the falling distance for\\s+t\\s*=\\s*([0-9.]+s?)",
        ]
        for pat in patterns:
            m = re.search(pat, prompt, re.IGNORECASE)
            if m: return m.group(1).strip()
        lines = [l.strip() for l in prompt.split("\\n") if l.strip()]
        for line in reversed(lines):
            if not line.startswith("In Alice") and " -> " not in line and " becomes " not in line:
                if ":" in line: return line.split(":", 1)[1].strip()
                return line
        return ""

    def _format_number(value, examples):
        decimals = []
        for _, out in examples:
            m = re.search(r"([0-9]+\\.([0-9]+))", out)
            if m: decimals.append(len(m.group(2)))
        precision = max(set(decimals), key=decimals.count) if decimals else 0
        if precision == 0: return f"{int(round(value))}"
        return f"{value:.{precision}f}"

    def solve_gravity(examples, test_t):
        ts, ds = [], []
        for t_str, d_str in examples:
            try:
                t = float(re.search(r"([0-9.]+)", t_str).group(1))
                d = float(re.search(r"([0-9.]+)", d_str).group(1))
                ts.append(t); ds.append(d)
            except: pass
        if not ts or len(ts) < 2: return "0.0"
        xs = [0.5 * t * t for t in ts]
        sum_xy = sum(d * x for d, x in zip(ds, xs))
        sum_x2 = sum(x * x for x in xs)
        g_ls = sum_xy / sum_x2 if sum_x2 != 0 else 0.0
        best_g, best_err = g_ls, float("inf")
        candidates = [g_ls] + [2*d/(t*t) for t,d in zip(ts,ds) if t>0]
        for g0 in candidates:
            for step in [0.01, 0.005, 0.002, 0.001]:
                for offset in range(-5, 6):
                    g = g0 + offset * step
                    err = sum((0.5 * g * t * t - d) ** 2 for t, d in zip(ts, ds))
                    if err < best_err: best_err, best_g = err, g
        try:
            test_t_val = float(re.search(r"([0-9.]+)", test_t).group(1))
            result = 0.5 * best_g * test_t_val * test_t_val
            return _format_number(result, examples)
        except: return "0.0"

    def solve_unit_conversion(examples, test_x):
        xs, ys = [], []
        for x_str, y_str in examples:
            try:
                x = float(re.search(r"([0-9.]+)", x_str).group(1))
                y = float(re.search(r"([0-9.]+)", y_str).group(1))
                xs.append(x); ys.append(y)
            except: pass
        if len(xs) < 2: return "0.0"
        n = len(xs)
        sum_x, sum_y = sum(xs), sum(ys)
        sum_xy = sum(x*y for x,y in zip(xs,ys))
        sum_x2 = sum(x*x for x in xs)
        denom = n * sum_x2 - sum_x * sum_x
        if abs(denom) < 1e-10:
            k = sum_y / sum_x if sum_x != 0 else 1.0
            try: test_val = float(re.search(r"([0-9.]+)", test_x).group(1)); return f"{k * test_val:.2f}"
            except: return "0.0"
        a = (n * sum_xy - sum_x * sum_y) / denom
        b = (sum_y - a * sum_x) / n
        try: test_val = float(re.search(r"([0-9.]+)", test_x).group(1)); return f"{a * test_val + b:.2f}"
        except: return "0.0"

    def int_to_roman(n):
        val = [1000, 900, 500, 400, 100, 90, 50, 40, 10, 9, 5, 4, 1]
        syms = ["M", "CM", "D", "CD", "C", "XC", "L", "XL", "X", "IX", "V", "IV", "I"]
        result = ""
        for v, s in zip(val, syms):
            while n >= v: result += s; n -= v
        return result

    def solve_numeral(examples, test_n):
        try: n = int(re.search(r"([0-9]+)", test_n).group(1))
        except: return ""
        return int_to_roman(n)

    def solve_bit_manip(examples, test_in):
        pairs = []
        for inp, out in examples:
            if len(inp) == 8 and set(inp).issubset({"0", "1"}):
                try: pairs.append((int(inp, 2), int(out, 2)))
                except: pass
        if not pairs: return test_in
        mapping = {}
        for out_bit in range(8):
            for in_bit in range(8):
                ok = all(((b >> out_bit) & 1) == ((a >> in_bit) & 1) for a, b in pairs)
                if ok: mapping[out_bit] = ("bit", in_bit, False); break
            if out_bit not in mapping:
                ok = all(((b >> out_bit) & 1) == 0 for a, b in pairs)
                if ok: mapping[out_bit] = ("const", 0, False)
                else:
                    ok = all(((b >> out_bit) & 1) == 1 for a, b in pairs)
                    if ok: mapping[out_bit] = ("const", 1, False)
        if len(mapping) == 8:
            try:
                test_val = int(test_in, 2); result = 0
                for out_bit in range(8):
                    typ, val, invert = mapping[out_bit]
                    bit = val if typ == "const" else ((test_val >> val) & 1)
                    if invert: bit = 1 - bit
                    result |= (bit << out_bit)
                return f"{result:08b}"
            except: pass
        for xor_const in range(256):
            if all((a ^ xor_const) == b for a, b in pairs):
                try: return f"{(int(test_in, 2) ^ xor_const):08b}"
                except: pass
        for and_const in range(256):
            if all((a & and_const) == b for a, b in pairs):
                try: return f"{(int(test_in, 2) & and_const):08b}"
                except: pass
        return test_in

    def solve_encryption(examples, test_in):
        mapping = {}
        for inp, out in examples:
            iw, ow = inp.split(), out.split()
            if len(iw) == len(ow):
                for a, b in zip(iw, ow):
                    if len(a) == len(b):
                        for c_in, c_out in zip(a, b):
                            if c_in not in mapping: mapping[c_in] = c_out
        result = " ".join("".join(mapping.get(c, "?") for c in w) for w in test_in.split())
        return result

    def solve_equations(examples, test_in):
        all_in = set(); all_out = set()
        for inp, out in examples:
            all_in.update(set(inp)); all_out.update(set(out))
        delete_set = all_in - all_out
        if delete_set:
            pred = "".join(c for c in test_in if c not in delete_set)
            if all("".join(c for c in inp if c not in delete_set) == out for inp, out in examples):
                return pred
        return ""

    def solve(prompt):
        ptype = classify_problem(prompt)
        test_input = extract_test_input(prompt)
        examples = parse_examples(prompt)
        if ptype == "gravity": return solve_gravity(examples, test_input)
        elif ptype == "unit_conversion": return solve_unit_conversion(examples, test_input)
        elif ptype == "numeral": return solve_numeral(examples, test_input)
        elif ptype == "bit_manip": return solve_bit_manip(examples, test_input)
        elif ptype == "encryption": return solve_encryption(examples, test_input)
        elif ptype == "equations": return solve_equations(examples, test_input)
        return ""

    print("[2/6] Symbolic solver loaded")

    # ── Load Training Data ──
    print("\n[3/6] Loading training data...")
    train_file = None
    for base_path in ["/kaggle/input/nvidia-nemotron-model-reasoning-challenge", "/kaggle/input"]:
        if os.path.exists(base_path):
            for root, dirs, files in os.walk(base_path):
                for f in files:
                    if f.lower() == 'train.csv':
                        train_file = os.path.join(root, f)
                        break
                if train_file: break
        if train_file: break

    if not train_file:
        raise FileNotFoundError("train.csv not found")

    df = pd.read_csv(train_file)
    print(f"  Loaded {len(df)} examples")

    # ── Generate Training CoT Data ──
    print("\n[4/6] Generating chain-of-thought training data...")
    training_texts = []
    correct = 0
    for idx, row in df.iterrows():
        prompt = row['prompt']
        answer = str(row['answer']).strip()
        pred = solve(prompt)
        is_correct = (pred.strip() == answer.strip())
        if is_correct: correct += 1

        ptype = classify_problem(prompt)
        trace_map = {
            "bit_manip": "This is a bit manipulation problem. Analyzing the input-output pairs reveals the bit transformation pattern.",
            "gravity": "Using physics formula d = 0.5 * g * t^2. Determining g from examples and calculating the falling distance.",
            "unit_conversion": "Finding the linear conversion factor from example pairs and applying it to the test measurement.",
            "numeral": "Converting the decimal number to Roman numerals using standard conversion rules.",
            "encryption": "Mapping character substitutions from the examples to decrypt the cipher text.",
            "equations": "Identifying the transformation rule from the examples and applying it to find the result.",
        }
        trace = trace_map.get(ptype, "Analyzing the pattern from the examples.")
        text = f"Solve this reasoning problem.\n\n{prompt}\n\n{trace}\n\nTherefore, the answer is \\boxed{{{answer}}}."
        training_texts.append(text)

        if (idx + 1) % 2000 == 0:
            print(f"  {idx+1}/{len(df)} ({correct} correct, {correct/(idx+1)*100:.1f}%)")

    print(f"\n  Symbolic solver accuracy: {correct}/{len(df)} ({correct/len(df)*100:.1f}%)")

    # ── Load Model + Tokenize ──
    print("\n[5/6] Loading Nemotron base model...")
    model_path = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_path, device_map="auto", trust_remote_code=True, torch_dtype=torch.bfloat16
    )

    lora_config = LoraConfig(
        r=32, lora_alpha=16,
        target_modules=["in_proj", "out_proj", "up_proj", "down_proj"],
        lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    # Tokenize
    print("  Tokenizing...")
    tokenized = []
    for text in training_texts:
        tokens = tokenizer(text, truncation=True, max_length=1024, padding=False)
        tokenized.append({"input_ids": tokens["input_ids"], "attention_mask": tokens["attention_mask"]})

    dataset = Dataset.from_list(tokenized)
    print(f"  Dataset ready: {len(dataset)} examples")

    # ── Training ──
    print("\n[6/6] Training LoRA adapter...")
    training_args = TrainingArguments(
        output_dir="/kaggle/working/nemotron_lora_adapter",
        num_train_epochs=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,
        learning_rate=1e-4,
        bf16=True,
        gradient_checkpointing=True,
        logging_steps=100,
        save_strategy="no",
        report_to="none",
    )

    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    trainer = Trainer(
        model=model, args=training_args,
        train_dataset=dataset,
        data_collator=data_collator,
    )

    trainer.train()

    # ── Save adapter explicitly ──
    adapter_dir = "/kaggle/working/nemotron_lora_adapter"
    model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)

    # Verify adapter_config.json exists
    config_path = os.path.join(adapter_dir, "adapter_config.json")
    if not os.path.exists(config_path):
        raise FileNotFoundError(f"adapter_config.json not found at {config_path}")
    print(f"  adapter_config.json exists: {os.path.getsize(config_path)} bytes")

    # Create submission.zip in /kaggle/working/
    import zipfile
    submission_path = "/kaggle/working/submission.zip"
    with zipfile.ZipFile(submission_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, files in os.walk(adapter_dir):
            for fname in files:
                fpath = os.path.join(root, fname)
                arcname = os.path.relpath(fpath, adapter_dir)
                zf.write(fpath, arcname)
                print(f"  Added to zip: {arcname} ({os.path.getsize(fpath)} bytes)")

    print(f"\n  Submission zip: {submission_path} ({os.path.getsize(submission_path)} bytes)")

    # Final verification
    with zipfile.ZipFile(submission_path, "r") as zf:
        names = zf.namelist()
        has_config = any("adapter_config" in n for n in names)
        print(f"  Zip contents: {names}")
        print(f"  adapter_config present: {has_config}")
        if not has_config:
            raise RuntimeError("adapter_config.json missing from submission.zip!")

    print("\n" + "=" * 60)
    print("SUBMISSION READY: /kaggle/working/submission.zip")
    print("=" * 60)

except Exception as e:
    print(f"\nERROR: {e}")
    traceback.print_exc()
    sys.exit(1)